# CP-SAT 的基本 API：写约束要用的那套工具

## 这一章讲什么

CP-SAT 常用的 API 一份清单，每一项都配一个能跑的示例。

- 变量怎么造（整数、布尔、跳着取值的）
- 线性约束，以及带条件的约束（`only_enforce_if`）
- 最大、最小、绝对值，还有乘、除、取余
- 逻辑约束（AND / OR / 恰好一个 / XOR）
- 允许和禁止的取值组合，数组元素引用（`add_element`），逆映射（`add_inverse`）

In [1]:
from ortools.sat.python import cp_model
from helper.ortools import get_all_solutions, is_between, is_not_between

## 这一章用到的函数

下表按用途把 API 分了类，每个函数在正文里都配了示例，可以当字典翻。示例一律用 `get_all_solutions()` 把**所有解**取出来摆在那儿，约束到底做了什么，不必猜，看解就明白。

| 分类 | 函数 |
|---|---|
| **创建变量** | `new_int_var`, `new_bool_var`, `new_int_var_from_domain` |
| **目标函数** | `minimize`, `maximize` |
| **线性约束** | `add`, `add_linear_constraint`, `add_linear_expression_in_domain` |
| **最大、最小、绝对值** | `add_min_equality`, `add_max_equality`, `add_abs_equality` |
| **乘、除、取余** | `add_modulo_equality`, `add_multiplication_equality`, `add_division_equality` |
| **互不相同的值** | `add_all_different` |
| **逻辑约束** | `add_implication`, `add_bool_or`, `add_at_least_one`, `add_at_most_one`, `add_exactly_one`, `add_bool_and`, `add_bool_xor` |
| **取值组合** | `add_allowed_assignments`, `add_forbidden_assignments` |
| **数组、元素** | `add_element`, `add_inverse` |

## 创建变量

- `new_int_var`: 给最小值和最大值，再给个变量名，整数变量就有了
- `new_bool_var`: 造一个最小值 0、最大值 1 的整数变量，参数只填变量名
- `new_int_var_from_domain`: 取值范围不连续的时候用它

下面照着这三个函数各走一遍。例子里都用辅助函数 `get_all_solutions()` 把解全捞出来，看看变量到底能取哪些值。

第一个例子，$0 \le x \le 3$ 的整数变量。

In [2]:
model = cp_model.CpModel()
x = model.new_int_var(0, 3, 'x')
get_all_solutions(model)

[{'x': 0}, {'x': 1}, {'x': 2}, {'x': 3}]

再看布尔那一路：$0 \le b \le 1$ 的整数变量，也就是布尔变量。

In [3]:
model = cp_model.CpModel()
x = model.new_bool_var('b')
get_all_solutions(model)

[{'b': 0}, {'b': 1}]

候选值想自己一个个点出来，就用 `Domain.from_values()`。

In [4]:
model = cp_model.CpModel()
domain = cp_model.Domain.from_values([1, 3, 5, 7])
x = model.new_int_var_from_domain(domain, 'x')
get_all_solutions(model)

[{'x': 1}, {'x': 3}, {'x': 5}, {'x': 7}]

范围不止一段时，交给 `Domain.from_flat_intervals()` 来指定。每一段都拿两个连续的值（先是下限、后是上限）组成一对写进去；哪一头没有设界限，就用 `INT_MIN` 和 `INT_MAX` 顶上。

In [5]:
model = cp_model.CpModel()
domain = cp_model.Domain.from_flat_intervals([1, 3, 7, 9, 10, 12])
x = model.new_int_var_from_domain(domain, 'x')
get_all_solutions(model)

[{'x': 1},
 {'x': 2},
 {'x': 3},
 {'x': 7},
 {'x': 8},
 {'x': 9},
 {'x': 10},
 {'x': 11},
 {'x': 12}]

## 线性约束

- `add()`: 把线性表达式的比较式直接当约束条件加进去。
- `add_linear_constraint()`: 连下限、上限一起给，加一条约束条件。
- `add_linear_expression_in_domain()`: 把线性表达式限制在不连续的范围里。

下面两个例子做的都是 $30 \le 10 x \le 50$ 这条线性约束，差别只在写法。例 1 用两个 `add()`，一头管下限，一头管上限；例 2 用 `add_linear_constraint()`，上下限一次说完。

In [6]:
model = cp_model.CpModel()
x = model.new_int_var(0, 100, 'x')
model.add(10 * x <= 50)
model.add(10 * x >= 30)
get_all_solutions(model)

[{'x': 3}, {'x': 4}, {'x': 5}]

In [7]:
model = cp_model.CpModel()
x = model.new_int_var(0, 100, 'x')
model.add_linear_constraint(10 * x, 30, 50)
get_all_solutions(model)

[{'x': 3}, {'x': 4}, {'x': 5}]

要把 $30 \leq 10x \leq 50$ **或者** $70 \leq 10x \leq 80$ 写成一条约束，就轮到 `add_linear_expression_in_domain()` 了。`Domain.from_flat_intervals([30, 50, 70, 80])` 摆的是两个区间：“30～50”和“70～80”。

In [8]:
model = cp_model.CpModel()
x = model.new_int_var(0, 100, 'x')
domain = cp_model.Domain.from_flat_intervals([30, 50, 70, 80])
model.add_linear_expression_in_domain(10 * x, domain)
get_all_solutions(model)

[{'x': 3}, {'x': 4}, {'x': 5}, {'x': 7}, {'x': 8}]

本书另外备了两个辅助函数，它们生成的是比较式，拿 `add()` 一加就成约束条件。

- $l \le x \le u$: `is_between(x, l, u)`
- $x < l \lor x > u$: `is_not_between(x, l, u)`

In [9]:
model = cp_model.CpModel()
x = model.new_int_var(1, 6, 'x')
y = model.new_int_var(1, 6, 'y')
model.add(is_between(x + y, 9, 10))
get_all_solutions(model)

[{'x': 3, 'y': 6},
 {'x': 4, 'y': 5},
 {'x': 4, 'y': 6},
 {'x': 5, 'y': 4},
 {'x': 5, 'y': 5},
 {'x': 6, 'y': 3},
 {'x': 6, 'y': 4}]

In [10]:
model = cp_model.CpModel()
x = model.new_int_var(1, 6, 'x')
y = model.new_int_var(1, 6, 'y')
model.add(is_not_between(x + y, 4, 11))
get_all_solutions(model)

[{'x': 1, 'y': 1}, {'x': 2, 'y': 1}, {'x': 1, 'y': 2}, {'x': 6, 'y': 6}]

## 条件分支

约束建好之后，只要调用 `only_enforce_if()`，就能指定它什么时候才生效。前提只能是布尔变量（一个或多个），或者它们的否定。比如要实现下面这个条件，就得中间垫一个布尔变量，把两个比较式拴在一块儿。

$$
1 \leq x \leq 3, \quad 1 \leq y \leq 3
$$
$$
z =
\begin{cases}
50, & \text{if } x + y \geq 4 \\
-50, & \text{otherwise}
\end{cases}
$$

代码里用 `only_enforce_if(cond)` 和 `only_enforce_if(~cond)` 这两个方法，把 `cond` 挂到 `x + y >= 4` 上，然后借这个条件去管 `z` 的取值。

In [11]:
model = cp_model.CpModel()
x, y = [model.new_int_var(1, 3, name) for name in 'xy']
z= model.new_int_var(-100, 100, 'z')

cond = model.new_bool_var('cond')
model.add(x + y >= 4).only_enforce_if(cond)
model.add(x + y < 4).only_enforce_if(~cond)

model.add(z == 50).only_enforce_if(cond)
model.add(z == -50).only_enforce_if(~cond)
get_all_solutions(model)

[{'x': 1, 'y': 1, 'z': -50, 'cond': 0},
 {'x': 2, 'y': 1, 'z': -50, 'cond': 0},
 {'x': 1, 'y': 2, 'z': -50, 'cond': 0},
 {'x': 1, 'y': 3, 'z': 50, 'cond': 1},
 {'x': 2, 'y': 2, 'z': 50, 'cond': 1},
 {'x': 3, 'y': 2, 'z': 50, 'cond': 1},
 {'x': 3, 'y': 3, 'z': 50, 'cond': 1},
 {'x': 2, 'y': 3, 'z': 50, 'cond': 1},
 {'x': 3, 'y': 1, 'z': 50, 'cond': 1}]

上面那种写法，比较式的否定得自己动手造，挺费事。本书的辅助函数 `equals(cond, expr, model)` 就是替你做这件事的：`expr` 的否定它自动生成，然后往 `model` 里加两条约束。

```python
model.add(expr).only_enforce_if(cond)
model.add(inverse_expr).only_enforce_if(~cond)
```

下面的代码用 `equals()` 把 `x + y >= 4` 这个条件挂到 `cond` 上。

In [12]:
from helper.ortools import equals
model = cp_model.CpModel()
x, y = [model.new_int_var(1, 3, name) for name in 'xy']
z= model.new_int_var(-100, 100, 'z')

cond = model.new_bool_var('cond')
equals(cond, x + y >= 4, model)

model.add(z == 50).only_enforce_if(cond)
model.add(z == -50).only_enforce_if(~cond)
get_all_solutions(model)

[{'x': 1, 'y': 1, 'z': -50, 'cond': 0},
 {'x': 2, 'y': 1, 'z': -50, 'cond': 0},
 {'x': 1, 'y': 2, 'z': -50, 'cond': 0},
 {'x': 1, 'y': 3, 'z': 50, 'cond': 1},
 {'x': 2, 'y': 2, 'z': 50, 'cond': 1},
 {'x': 3, 'y': 2, 'z': 50, 'cond': 1},
 {'x': 3, 'y': 3, 'z': 50, 'cond': 1},
 {'x': 2, 'y': 3, 'z': 50, 'cond': 1},
 {'x': 3, 'y': 1, 'z': 50, 'cond': 1}]

这段代码要的是“取小的那一个”：`2 * x` 和 `y` 里较小的那个等于 2。比如 `x=1, y=3`，`min(2, 3) = 2`，这就成了一个解。

In [13]:
model = cp_model.CpModel()
x, y = [model.new_int_var(0, 4, n) for n in 'xy']
model.add_min_equality(2, [2 * x,  y])
get_all_solutions(model)

[{'x': 1, 'y': 2},
 {'x': 1, 'y': 3},
 {'x': 1, 'y': 4},
 {'x': 2, 'y': 2},
 {'x': 3, 'y': 2},
 {'x': 4, 'y': 2}]

这段换成了大的那个：`2 * x` 和 `y` 中较大者必须是 4。像 `x=2, y=3`，`max(4, 3) = 4`，这种组合就是解。

In [14]:
model = cp_model.CpModel()
x, y = [model.new_int_var(1, 4, n) for n in 'xy']
model.add_max_equality(4, [2 * x, y])
get_all_solutions(model)

[{'x': 1, 'y': 4},
 {'x': 2, 'y': 1},
 {'x': 2, 'y': 2},
 {'x': 2, 'y': 3},
 {'x': 2, 'y': 4}]

这次盯的是绝对值：`x - y` 的绝对值必须等于 `2`。像 `x=3, y=1` 这种组合，`|3 - 1| = 2`，于是它也是解。

In [15]:
model = cp_model.CpModel()
x, y = [model.new_int_var(1, 4, n) for n in 'xy']
model.add_abs_equality(2, x - y)
get_all_solutions(model)

[{'x': 1, 'y': 3}, {'x': 2, 'y': 4}, {'x': 3, 'y': 1}, {'x': 4, 'y': 2}]

## 变量之间的乘法和除法

整数变量之间要写乘法、除法，有几个省事的方法可用，下面这三个都是现成的。

* `add_modulo_equality(target, dividend, divisor)`: 添加 `dividend % divisor == target` 这个约束
* `add_multiplication_equality(target, *expressions)`: 添加 `product(expressions) == target` 这个约束
* `add_division_equality(target, numerator, denominator)`: 添加 `numerator // denominator == target` 这个约束（整数除法）

下面这段代码用 `add_multiplication_equality()` 找三边长都是整数的直角三角形，也就是勾股数。

In [16]:
model = cp_model.CpModel()
max_n = 20
x, y, z = [model.new_int_var(1, max_n, n) for n in 'xyz']
x2, y2, z2 = [model.new_int_var(1, 2 * max_n**2, n) for n in ['x2', 'y2', 'z2']]
model.add_multiplication_equality(x2, x, x)
model.add_multiplication_equality(y2, y, y)
model.add_multiplication_equality(z2, z, z)
model.add(x2 + y2 == z2)
model.add(y >= x)
get_all_solutions(model)

[{'x': 3, 'y': 4, 'z': 5, 'x2': 9, 'y2': 16, 'z2': 25},
 {'x': 5, 'y': 12, 'z': 13, 'x2': 25, 'y2': 144, 'z2': 169},
 {'x': 6, 'y': 8, 'z': 10, 'x2': 36, 'y2': 64, 'z2': 100},
 {'x': 8, 'y': 15, 'z': 17, 'x2': 64, 'y2': 225, 'z2': 289},
 {'x': 9, 'y': 12, 'z': 15, 'x2': 81, 'y2': 144, 'z2': 225},
 {'x': 12, 'y': 16, 'z': 20, 'x2': 144, 'y2': 256, 'z2': 400}]

## 互不相同的值

要让一堆变量彼此都不相等，有现成的 `add_all_different(*expressions)`。它给指定的所有表达式都加上“取值互不相同”这条约束，数独、分配问题这类非要不重样的地方，常常见得到它。下面的代码求的是 `1, 2, 3` 的排列。

In [17]:
model = cp_model.CpModel()
x, y, z = [model.new_int_var(1, 3, n) for n in 'xyz']
model.add_all_different(x, y, z)
get_all_solutions(model)

[{'x': 3, 'y': 2, 'z': 1},
 {'x': 3, 'y': 1, 'z': 2},
 {'x': 2, 'y': 3, 'z': 1},
 {'x': 2, 'y': 1, 'z': 3},
 {'x': 1, 'y': 3, 'z': 2},
 {'x': 1, 'y': 2, 'z': 3}]

## 逻辑约束

这一节的约束只认**布尔变量和它的否定**，整数变量塞不进去，这点得留意。

| 方法 | 内容 |
|----------|------|
| `add_implication(a, b)` | **a ⇒ b**（a 为真则 b 也为真） |
| `add_bool_or([a, b, c])` | **a OR b OR c**（至少 1 个为真） |
| `add_at_least_one([a, b, c])` | 至少 1 个为真 |
| `add_at_most_one([a, b, c])` | 最多 1 个为真 |
| `add_exactly_one([a, b, c])` | 恰好 1 个为真 |
| `add_bool_and([a, b, c])` | **a AND b AND c**（全部为真） |
| `add_bool_xor([a, b, c])` | 为真的变量个数是奇数 |

`add_implication(a, b)` 加的是逻辑蕴含，说白了就是“a 为真，b 也得为真”。这条约束可以用下面这张表说清楚。

| 输入 a | 输入 b | 输出 a → b |
|--------|--------|------------|
|   0    |   0    |     1      |
|   0    |   1    |     1      |
|   1    |   0    |     0      |
|   1    |   1    |     1      |

只有 `a` 为真、`b` 也为真的时候，“输出 a → b”才是 1。下面的代码去找“输出 a → b”为 1 的解，一共输出 3 个解。

In [18]:
model = cp_model.CpModel()
a, b = [model.new_bool_var(n) for n in 'ab']
model.add_implication(a, b)
get_all_solutions(model)

[{'a': 0, 'b': 0}, {'a': 0, 'b': 1}, {'a': 1, 'b': 1}]

`add_bool_or()` 和 `add_at_least_one()` 是一回事：布尔变量列表里至少得有一个为真。换句话说，全假的那种组合被排除了。

In [19]:
model = cp_model.CpModel()
x, y, z = [model.new_bool_var(n) for n in 'xyz']
model.add_bool_or(x, y, z)
get_all_solutions(model)

[{'x': 0, 'y': 1, 'z': 0},
 {'x': 0, 'y': 1, 'z': 1},
 {'x': 0, 'y': 0, 'z': 1},
 {'x': 1, 'y': 0, 'z': 1},
 {'x': 1, 'y': 0, 'z': 0},
 {'x': 1, 'y': 1, 'z': 0},
 {'x': 1, 'y': 1, 'z': 1}]

`add_at_most_one()` 管的是另一头：布尔变量列表里最多一个为真，也就是说，两个以上同时为真这种事不会发生。

In [20]:
model = cp_model.CpModel()
x, y, z = [model.new_bool_var(n) for n in 'xyz']
model.add_at_most_one(x, y, z)
get_all_solutions(model)

[{'x': 0, 'y': 0, 'z': 0},
 {'x': 0, 'y': 1, 'z': 0},
 {'x': 0, 'y': 0, 'z': 1},
 {'x': 1, 'y': 0, 'z': 0}]

`add_exactly_one()` 要的是恰好一个：布尔变量列表里不多不少，正好 1 个为真。

In [21]:
model = cp_model.CpModel()
x, y, z = [model.new_bool_var(n) for n in 'xyz']
model.add_exactly_one(x, y, z)
get_all_solutions(model)

[{'x': 0, 'y': 1, 'z': 0}, {'x': 0, 'y': 0, 'z': 1}, {'x': 1, 'y': 0, 'z': 0}]

`add_bool_and()` 把布尔变量列表整个按住，要求全部为真。

In [22]:
model = cp_model.CpModel()
x, y, z = [model.new_bool_var(n) for n in 'xyz']
model.add_bool_and(x, y, z)
get_all_solutions(model)

[{'x': 1, 'y': 1, 'z': 1}]

这条约束往往太强，所以一般配上 `only_enforce_if()` 一起用。

In [23]:
model = cp_model.CpModel()
x, y, z = [model.new_bool_var(n) for n in 'xyz']
model.add_bool_and(x, y).only_enforce_if(~z)
get_all_solutions(model)

[{'x': 1, 'y': 1, 'z': 0},
 {'x': 1, 'y': 1, 'z': 1},
 {'x': 1, 'y': 0, 'z': 1},
 {'x': 0, 'y': 0, 'z': 1},
 {'x': 0, 'y': 1, 'z': 1}]

`add_bool_xor()` 数的是个数：布尔变量列表里为真的变量，个数必须是奇数。

In [24]:
model = cp_model.CpModel()
x, y, z = [model.new_bool_var(n) for n in 'xyz']
model.add_bool_xor(x, y, z)
get_all_solutions(model)

[{'x': 0, 'y': 1, 'z': 0},
 {'x': 0, 'y': 0, 'z': 1},
 {'x': 1, 'y': 0, 'z': 0},
 {'x': 1, 'y': 1, 'z': 1}]

## 取值组合的解

`add_allowed_assignments()` 加的是允许这一面：给指定的变量列出可取的组合，它们就只能在这些组合里挑。列表里每个变量，都被按在特定的组合上。

下面的代码给 `x` 和 `y` 指定组合（`[5, 6]` 和 `[6, 7]`），给 `y` 和 `z` 另指定一组（`[6, 8]` 和 `[7, 9]`），然后求允许的解。

In [25]:
model = cp_model.CpModel()
x, y, z = [model.new_int_var(1, 10, n) for n in 'xyz']
model.add_allowed_assignments([x, y], [[5, 6], [6, 7]])
model.add_allowed_assignments([y, z], [[6, 8], [7, 9]])
get_all_solutions(model)

[{'x': 5, 'y': 6, 'z': 8}, {'x': 6, 'y': 7, 'z': 9}]

要给计算式指定候选解，中间变量就得登场了。下面代码里的 `x_plus_y` 和 `y_plus_z` 就是中间变量，分别存着 `x + y` 和 `y + z`。有了它们，`add_allowed_assignments()` 才能规定 `x_plus_y` 取 `5` 或 `6`、`y_plus_z` 取 `6` 或 `7` 这样的组合。复杂的计算式想定候选解，中间变量这一手最省事。

In [26]:
model = cp_model.CpModel()
x, y, z = [model.new_int_var(1, 10, n) for n in 'xyz']
x_plus_y = model.new_int_var(-100, 100, 'x + y')
y_plus_z = model.new_int_var(-100, 100, 'y + z')
model.add(x_plus_y == x + y)
model.add(y_plus_z == y + z)
model.add_allowed_assignments([x_plus_y, y_plus_z], [[5, 6], [6, 7]])
get_all_solutions(model)

[{'x': 1, 'y': 4, 'z': 2, 'x + y': 5, 'y + z': 6},
 {'x': 2, 'y': 3, 'z': 3, 'x + y': 5, 'y + z': 6},
 {'x': 3, 'y': 2, 'z': 4, 'x + y': 5, 'y + z': 6},
 {'x': 4, 'y': 1, 'z': 5, 'x + y': 5, 'y + z': 6},
 {'x': 4, 'y': 2, 'z': 5, 'x + y': 6, 'y + z': 7},
 {'x': 3, 'y': 3, 'z': 4, 'x + y': 6, 'y + z': 7},
 {'x': 2, 'y': 4, 'z': 3, 'x + y': 6, 'y + z': 7},
 {'x': 5, 'y': 1, 'z': 6, 'x + y': 6, 'y + z': 7},
 {'x': 1, 'y': 5, 'z': 2, 'x + y': 6, 'y + z': 7}]

`add_forbidden_assignments()` 站的是反面：把哪些组合不许出现一一指定下来，它们就真的不会出现在解里。下面的代码对 `x` 与 `y`、以及 `y` 与 `z` 这两对，把 `1, 2`、`1, 3`、`2, 2` 这几种组合统统禁掉了。

In [27]:
model = cp_model.CpModel()
x, y, z = [model.new_int_var(1, 3, n) for n in 'xyz']
model.add_forbidden_assignments([x, y], [[1, 2], [1, 3], [2, 2]])
model.add_forbidden_assignments([y, z], [[1, 2], [1, 3], [2, 2]])
get_all_solutions(model)

[{'x': 3, 'y': 3, 'z': 1},
 {'x': 3, 'y': 2, 'z': 1},
 {'x': 3, 'y': 1, 'z': 1},
 {'x': 2, 'y': 1, 'z': 1},
 {'x': 2, 'y': 3, 'z': 1},
 {'x': 2, 'y': 3, 'z': 2},
 {'x': 2, 'y': 3, 'z': 3},
 {'x': 3, 'y': 3, 'z': 3},
 {'x': 3, 'y': 2, 'z': 3},
 {'x': 3, 'y': 3, 'z': 2},
 {'x': 1, 'y': 1, 'z': 1}]

## 数组元素

`add_element(index, values, element)` 说的是：数组 `values` 里第 `index` 个元素，正好等于 `element`。想按位置把数组里某个元素钉在特定的值上，用它就行。

In [28]:
model = cp_model.CpModel()
index = model.new_int_var(0, 4, 'index')
element = model.new_int_var(-100, 100, 'element')
model.add_element(index, [1, 3, 5, 7], element)
get_all_solutions(model)

[{'index': 0, 'element': 1},
 {'index': 1, 'element': 3},
 {'index': 2, 'element': 5},
 {'index': 3, 'element': 7}]

`values` 也可以是变量列表，不限于常量列表。

In [29]:
model = cp_model.CpModel()
x, y, z =  [model.new_bool_var(n) for n in 'xyz']
index = model.new_int_var(0, 3, 'index')
element = model.new_int_var(-100, 100, 'element')
model.add_at_most_one([x, y, z])
model.add_element(index, [x, y, z], element)
get_all_solutions(model)

[{'x': 0, 'y': 0, 'z': 0, 'index': 0, 'element': 0},
 {'x': 0, 'y': 1, 'z': 0, 'index': 0, 'element': 0},
 {'x': 0, 'y': 0, 'z': 1, 'index': 0, 'element': 0},
 {'x': 0, 'y': 0, 'z': 1, 'index': 1, 'element': 0},
 {'x': 0, 'y': 0, 'z': 0, 'index': 1, 'element': 0},
 {'x': 0, 'y': 1, 'z': 0, 'index': 1, 'element': 1},
 {'x': 0, 'y': 0, 'z': 1, 'index': 2, 'element': 1},
 {'x': 0, 'y': 0, 'z': 0, 'index': 2, 'element': 0},
 {'x': 0, 'y': 1, 'z': 0, 'index': 2, 'element': 0},
 {'x': 1, 'y': 0, 'z': 0, 'index': 2, 'element': 0},
 {'x': 1, 'y': 0, 'z': 0, 'index': 1, 'element': 0},
 {'x': 1, 'y': 0, 'z': 0, 'index': 0, 'element': 1}]

`add_inverse(a, b)` 拿数组 `a` 的索引去定数组 `b` 的元素，具体约束的是 `b[a[i]] == i`。这样一来，`b` 必定跟 `a` 的 `arg_sort()` 对得上。

举个例子，`a` 的元素是 `[1, 2, 0]` 的时候，索引按 `[2, 0, 1]` 的顺序摆进 `b`，于是 `b[1] == 0`, `b[2] == 1`, `b[0] == 2`。

In [30]:
import numpy as np

model = cp_model.CpModel()
a = [model.new_int_var(0, 2, n) for n in ['a0', 'a1', 'a2']]
b = [model.new_int_var(0, 2, n) for n in ['b0', 'b1', 'b2']]
model.add_inverse(a, b)
for sol in get_all_solutions(model):
    a_sol = [sol[f'a{i}'] for i in range(3)]
    b_sol = [sol[f'b{i}'] for i in range(3)]
    print(f'a = {a_sol}, b = {b_sol}, arg_sort = {np.argsort(a_sol).tolist()}')

a = [2, 1, 0], b = [2, 1, 0], arg_sort = [2, 1, 0]
a = [1, 2, 0], b = [2, 0, 1], arg_sort = [2, 0, 1]
a = [2, 0, 1], b = [1, 2, 0], arg_sort = [1, 2, 0]
a = [1, 0, 2], b = [1, 0, 2], arg_sort = [1, 0, 2]
a = [0, 2, 1], b = [0, 2, 1], arg_sort = [0, 2, 1]
a = [0, 1, 2], b = [0, 1, 2], arg_sort = [0, 1, 2]


## 小结

| 想做的事 | 用的东西 |
|---|---|
| 整数变量 | `new_int_var(lb, ub, name)` |
| 0/1 变量 | `new_bool_var(name)` |
| 只用离散的值 | `new_int_var_from_domain(Domain.from_values([...]), name)` |
| 区间的集合 | `Domain.from_flat_intervals([lb1, ub1, lb2, ub2])` |
| 上下限一起设 | `add_linear_constraint(式, lb, ub)` |
| 收进不连续的范围 | `add_linear_expression_in_domain(式, domain)` |
| 有条件地让约束生效 | `model.add(...).only_enforce_if(b)` |
| 最大、最小、绝对值 | `add_max_equality` / `add_min_equality` / `add_abs_equality` |
| 积、商、余数 | `add_multiplication_equality` / `add_division_equality` / `add_modulo_equality` |
| 全部不同 | `add_all_different` |
| 允许、禁止组合 | `add_allowed_assignments` / `add_forbidden_assignments` |
| 引用数组 | `add_element(index, values, element)` |
| 逆映射 | `add_inverse(a, b)`（`b[a[i]] == i`） |

- 约束做的事，说到底就是不断砍掉解的候选。用 `get_all_solutions()` 把解集摊开看，它的意思就清楚了
- 逻辑约束只认布尔变量。想拿整数用，先把比较式挂到一个布尔变量上